# Session 05 Lab: Your First AI App
**Course:** Noob to AI Expert | **Track:** Beginner

Build a working AI Study Assistant with conversation history and streaming.

In [ ]:
!pip install anthropic -q
print('✅ Ready')

In [ ]:
import anthropic
import os

os.environ.setdefault('ANTHROPIC_API_KEY', 'your-key-here')
client = anthropic.Anthropic()

SYSTEM_PROMPT = """You are a patient AI tutor specializing in computer science and AI.

Guidelines:
- Explain concepts clearly with analogies and examples
- Ask clarifying questions to check understanding
- Build on previous messages in the conversation
- Encourage the student and celebrate progress
- Keep explanations concise (2-3 paragraphs max)
"""

print('System prompt configured.')

## Part 1: Single Turn

In [ ]:
def ask_tutor(question: str) -> str:
    resp = client.messages.create(
        model='claude-haiku-4-5-20251001',
        max_tokens=512,
        system=SYSTEM_PROMPT,
        messages=[{'role': 'user', 'content': question}]
    )
    return resp.content[0].text

print(ask_tutor('What is machine learning? Explain it simply.'))

## Part 2: Multi-Turn Conversation

In [ ]:
class StudyAssistant:
    def __init__(self):
        self.history = []

    def chat(self, user_message: str) -> str:
        self.history.append({'role': 'user', 'content': user_message})
        resp = client.messages.create(
            model='claude-haiku-4-5-20251001',
            max_tokens=512,
            system=SYSTEM_PROMPT,
            messages=self.history
        )
        reply = resp.content[0].text
        self.history.append({'role': 'assistant', 'content': reply})
        return reply

    def clear(self):
        self.history = []
        print('Conversation cleared.')

assistant = StudyAssistant()

conversation = [
    'What is a neural network?',
    'How is it different from regular programming?',
    'Can you give me a quiz question to test my understanding?'
]

for msg in conversation:
    print(f'You: {msg}')
    reply = assistant.chat(msg)
    print(f'Tutor: {reply[:300]}...')
    print()

## Part 3: Streaming Responses

In [ ]:
import sys

def stream_response(question: str):
    print('Tutor: ', end='', flush=True)
    with client.messages.stream(
        model='claude-haiku-4-5-20251001',
        max_tokens=300,
        system=SYSTEM_PROMPT,
        messages=[{'role': 'user', 'content': question}]
    ) as stream:
        for text in stream.text_stream:
            print(text, end='', flush=True)
    print()  # newline at end

stream_response('Explain gradient descent in 3 sentences.')